In [1]:
import pandas as pd
import nasdaqdatalink

nasdaqdatalink.read_key(r"key\secret.txt")

df = nasdaqdatalink.get_table(
    "WIKI/PRICES",
    ticker="AAPL",
    date={"gte": "1996-01-01", "lte": "2026-10-27"},
    paginate=True,
)
df = df.sort_values("date").reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"])
df.head()

,ticker,date,open,high,low,close,volume,ex-dividend,split_ratio,adj_open,adj_high,adj_low,adj_close,adj_volume
0,AAPL,1996-01-02,32.25,32.25,31.75,32.13,1243700.0,0.0,1.0,1.036144,1.036144,1.020079,1.032288,34823600.0
1,AAPL,1996-01-03,32.00,32.88,31.87,32.13,3837800.0,0.0,1.0,1.028112,1.056385,1.023935,1.032288,107458400.0
2,AAPL,1996-01-04,32.38,32.38,31.37,31.56,2680200.0,0.0,1.0,1.040320,1.040320,1.007871,1.013975,75045600.0
3,AAPL,1996-01-05,31.62,34.25,31.37,34.25,3981500.0,0.0,1.0,1.015903,1.100401,1.007871,1.100401,111482000.0
4,AAPL,1996-01-08,34.50,35.50,34.00,34.63,1083400.0,0.0,1.0,1.108433,1.140561,1.092368,1.112609,30335200.0


In [2]:
import pandas as pd

df_cpi = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL")
df_cpi.head()

,observation_date,CPIAUCSL
0,1947-01-01,21.48
1,1947-02-01,21.62
2,1947-03-01,22.00
3,1947-04-01,22.00
4,1947-05-01,21.95


In [3]:
monthly = df.set_index("date")["adj_close"].resample("MS").last().to_frame("adj_close")

cpi = df_cpi.rename(columns={"observation_date": "date", "CPIAUCSL": "cpi"})
cpi["date"] = pd.to_datetime(cpi["date"])
cpi = cpi.set_index("date")

data = monthly.join(cpi, how="inner")
data = data.reset_index()
data.head()

,date,adj_close,cpi
0,1996-01-01,0.887389,154.7
1,1996-02-01,0.883533,155.0
2,1996-03-01,0.789076,155.5
3,1996-04-01,0.782971,156.1
4,1996-05-01,0.839196,156.4


In [4]:
import numpy as np

data["simple_return"] = data["adj_close"].pct_change()
data["log_return"] = np.log(data["adj_close"] / data["adj_close"].shift(1))
data["inflation"] = data["cpi"].pct_change()
data["real_return"] = (1 + data["simple_return"]) / (1 + data["inflation"]) - 1

data.head()

,date,adj_close,cpi,simple_return,log_return,inflation,real_return
0,1996-01-01,0.887389,154.7,NaN,NaN,NaN,NaN
1,1996-02-01,0.883533,155.0,-0.004345,-0.004354,0.001939,-0.006272
2,1996-03-01,0.789076,155.5,-0.106909,-0.113067,0.003226,-0.109781
3,1996-04-01,0.782971,156.1,-0.007736,-0.007766,0.003859,-0.011550
4,1996-05-01,0.839196,156.4,0.071810,0.069348,0.001922,0.069754
